# 20.08 - Instance segmentation data and stratified folds

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** File-backed instance masks, a persisted three-fold table, and a Torchvision-ready dataset.

The competition boundary is the dataset and fold table: stable IDs make leakage checks, OOF predictions, and reruns auditable.

## Core ideas

- Semantic segmentation assigns one class per pixel; instance segmentation also separates objects of the same class.
- Mask R-CNN consumes a list of float images `[C,H,W]` in `[0,1]`. During training each target needs `boxes`, `labels`, and binary instance `masks`.
- A single image can contain several classes, so plain single-label stratification is not exact. This toy dataset uses the documented proxy `class composition + instance-count bin`. In a real grouped or multilabel dataset, switch to a group-aware or iterative multilabel method.
- Split before augmentation. Persist `image_id -> fold`, train on folds `!= k`, validate only on fold `k`, and never duplicate validation images.

In [ ]:
import csv
import json
import os

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import tv_tensors
from torchvision.ops import masks_to_boxes
from torchvision.transforms import v2

SEED = 20
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_DIR = "_day20_instance_data"
MANIFEST_PATH = os.path.join(DATA_DIR, "manifest.csv")
FOLD_PATH = os.path.join(DATA_DIR, "folds.csv")


## Prepared image data

The provided generator writes 18 RGB images, integer instance-ID masks, and a CSV manifest. It is deliberately complete; data generation is not an exercise.

**Return structure — `generate_instance_fixture`:** A `dict` with `data_dir` (`str`), `manifest_path` (`str`), `images` (`int`), and `strata` (`int`). It creates `images/*.png`, `masks/*.png`, and `manifest.csv` under `data_dir`. Each manifest row stores one stable `image_id`, integer `sample_index`, relative file paths, JSON `instance_labels` ordered by nonzero mask ID, and string `split_key`.

In [ ]:
def generate_instance_fixture(data_dir, samples_per_stratum=6, image_size=96, seed=20):
    image_dir = os.path.join(data_dir, "images")
    mask_dir = os.path.join(data_dir, "masks")
    manifest_path = os.path.join(data_dir, "manifest.csv")
    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(mask_dir, exist_ok=True)
    rng = np.random.default_rng(seed)
    rows = []
    sample_index = 0

    for split_key in ("circle_1", "square_1", "mixed_2"):
        for local_index in range(int(samples_per_stratum)):
            image_id = f"image_{sample_index:03d}"
            pixels = rng.integers(18, 36, size=(image_size, image_size, 3), dtype=np.uint8)
            image = Image.fromarray(pixels, mode="RGB")
            instance_map = Image.fromarray(np.zeros((image_size, image_size), dtype=np.uint8), mode="L")
            image_draw = ImageDraw.Draw(image)
            mask_draw = ImageDraw.Draw(instance_map)
            labels = []

            if split_key in ("circle_1", "mixed_2"):
                offset = int(local_index % 5)
                circle_box = (8 + offset, 10, 38 + offset, 40)
                image_draw.ellipse(circle_box, fill=(220, 70, 70), outline=(255, 220, 220), width=2)
                mask_draw.ellipse(circle_box, fill=1)
                labels.append(1)

            if split_key in ("square_1", "mixed_2"):
                offset = int((local_index * 3) % 7)
                square_box = (53 - offset, 51, 84 - offset, 82)
                instance_id = 1 if split_key == "square_1" else 2
                image_draw.rectangle(square_box, fill=(70, 170, 235), outline=(220, 245, 255), width=2)
                mask_draw.rectangle(square_box, fill=instance_id)
                labels.append(2)

            image_path = os.path.join(image_dir, image_id + ".png")
            mask_path = os.path.join(mask_dir, image_id + ".png")
            image.save(image_path)
            instance_map.save(mask_path)
            rows.append(
                {
                    "sample_index": sample_index,
                    "image_id": image_id,
                    "image_path": image_path,
                    "mask_path": mask_path,
                    "instance_labels": json.dumps(labels),
                    "split_key": split_key,
                }
            )
            sample_index += 1

    with open(manifest_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return {
        "data_dir": data_dir,
        "manifest_path": manifest_path,
        "images": len(rows),
        "strata": 3,
    }

fixture_info = generate_instance_fixture(DATA_DIR)
print(fixture_info)


## Exercise 20-A: Persist deterministic stratified folds

Use the full manifest and `StratifiedKFold`, not repeated holdouts. The proxy has six examples per stratum, enough for three folds; every sample must be validation exactly once.

**Return structure — `build_fold_table`:** A `pandas.DataFrame` with all manifest columns plus integer `fold` in `[0,n_splits)`, sorted by `image_id`, with exactly one row per image. It also writes the same table to `output_path`.

In [ ]:
def build_fold_table(manifest_path, output_path, n_splits=3, seed=20):
    frame = pd.read_csv(manifest_path).sort_values("image_id").reset_index(drop=True)
    frame["fold"] = -1
    splitter = StratifiedKFold(n_splits=int(n_splits), shuffle=True, random_state=int(seed))
    for fold, (_, validation_indices) in enumerate(
        splitter.split(np.zeros(len(frame), dtype=np.uint8), frame["split_key"])
    ):
        frame.loc[validation_indices, "fold"] = int(fold)
    frame["fold"] = frame["fold"].astype(int)
    frame.to_csv(output_path, index=False)
    return frame

# Smoke check: build all folds and inspect per-fold proxy support.
fold_frame = build_fold_table(MANIFEST_PATH, FOLD_PATH, n_splits=3, seed=SEED)
print(pd.crosstab(fold_frame["fold"], fold_frame["split_key"]))


## Exercise 20-B: Decode instance targets safely

Use `masks_to_boxes` and Torchvision `v2` transforms so flips update images, masks, and boxes together. Only the training dataset receives augmentation.

**Return structure — `InstanceMaskDataset`:** A callable `torch.utils.data.Dataset`. `len(dataset)` is the selected row count. `dataset[i]` is a tuple: position 0 is a CPU `torch.float32` image `[3,H,W]` in `[0,1]`; position 1 is a dictionary with `boxes` (`float32 [N,4]`, XYXY), `labels` (`int64 [N]`, foreground IDs 1..C-1), `masks` (`uint8 [N,H,W]`), `image_id` (`int64 [1]`), `area` (`float32 [N]`), and `iscrowd` (`int64 [N]`). `N` is the image's instance count.

In [ ]:
class InstanceMaskDataset(Dataset):
    def __init__(self, fold_frame, validation_fold, split, augment=False):
        if split not in ("train", "validation"):
            raise ValueError("split must be 'train' or 'validation'")
        selector = fold_frame["fold"] != int(validation_fold)
        if split == "validation":
            selector = ~selector
        self.rows = fold_frame.loc[selector].sort_values("image_id").reset_index(drop=True)
        operations = [v2.RandomHorizontalFlip(p=0.5)] if augment else []
        operations.extend([v2.ToDtype(torch.float32, scale=True), v2.ToPureTensor()])
        self.transforms = v2.Compose(operations)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[int(index)]
        image_array = np.array(Image.open(row["image_path"]).convert("RGB"), copy=True)
        instance_array = np.array(Image.open(row["mask_path"]), copy=True)
        instance_ids = [int(value) for value in np.unique(instance_array) if int(value) != 0]
        masks = torch.stack(
            [torch.from_numpy((instance_array == instance_id).astype(np.uint8)) for instance_id in instance_ids]
        )
        labels = torch.tensor(json.loads(row["instance_labels"]), dtype=torch.int64)
        boxes = masks_to_boxes(masks)
        image = tv_tensors.Image(torch.from_numpy(image_array).permute(2, 0, 1))
        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=instance_array.shape),
            "labels": labels,
            "masks": tv_tensors.Mask(masks),
            "image_id": torch.tensor([int(row["sample_index"])], dtype=torch.int64),
            "area": masks.flatten(1).sum(1).to(torch.float32),
            "iscrowd": torch.zeros(len(instance_ids), dtype=torch.int64),
        }
        image, target = self.transforms(image, target)
        target["boxes"] = target["boxes"].to(torch.float32)
        target["masks"] = target["masks"].to(torch.uint8)
        return image, target

# Smoke check: decode one untouched validation image.
validation_dataset = InstanceMaskDataset(fold_frame, validation_fold=0, split="validation", augment=False)
smoke_image, smoke_target = validation_dataset[0]
print(smoke_image.shape, smoke_image.dtype, {key: tuple(value.shape) for key, value in smoke_target.items()})


## Exercise 20-C: Collate variable instance counts

Detection models accept lists/tuples because images and numbers of instances may differ. Do not use the default stacking collator.

**Return structure — `instance_collate`:** A tuple of length 2. Position 0 is a tuple of `B` image tensors; position 1 is a tuple of `B` target dictionaries. `B` is the input batch length.

In [ ]:
def instance_collate(batch):
    images, targets = zip(*batch)
    return tuple(images), tuple(targets)

# Smoke check: one batch preserves one target dictionary per image.
train_dataset = InstanceMaskDataset(fold_frame, validation_fold=0, split="train", augment=True)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=False, collate_fn=instance_collate)
batch_images, batch_targets = next(iter(train_loader))
print(len(batch_images), [int(target["labels"].numel()) for target in batch_targets])


## Test Cases

Tests cover persisted artifacts, fold separation and balance, image/target contracts, and the variable-length collator.

**Return structure — `run_day20_tests`:** Returns `None`. Assertions communicate failure; the exact line `Day 20 tests passed` communicates success.

In [ ]:
def run_day20_tests():
    assert os.path.isfile(MANIFEST_PATH) and os.path.isfile(FOLD_PATH)
    assert len(fold_frame) == 18 and fold_frame["image_id"].is_unique
    assert sorted(fold_frame["fold"].unique().tolist()) == [0, 1, 2]
    support = pd.crosstab(fold_frame["fold"], fold_frame["split_key"])
    assert support.shape == (3, 3) and (support.to_numpy() == 2).all()
    train_ids = set(fold_frame.loc[fold_frame["fold"] != 0, "image_id"])
    validation_ids = set(fold_frame.loc[fold_frame["fold"] == 0, "image_id"])
    assert train_ids.isdisjoint(validation_ids) and len(train_ids | validation_ids) == 18
    assert smoke_image.shape == (3, 96, 96) and smoke_image.dtype == torch.float32
    assert 0.0 <= float(smoke_image.min()) <= float(smoke_image.max()) <= 1.0
    required = {"boxes", "labels", "masks", "image_id", "area", "iscrowd"}
    assert set(smoke_target) == required
    count = int(smoke_target["labels"].numel())
    assert smoke_target["boxes"].shape == (count, 4) and smoke_target["boxes"].dtype == torch.float32
    assert smoke_target["masks"].shape == (count, 96, 96) and smoke_target["masks"].dtype == torch.uint8
    assert smoke_target["labels"].dtype == torch.int64 and set(smoke_target["labels"].tolist()) <= {1, 2}
    assert len(batch_images) == len(batch_targets) == 2
    print("Day 20 tests passed")

run_day20_tests()


## Day 20 Checklist

- [ ] Explain why an instance-ID mask differs from a semantic class mask.
- [ ] Persist stable three-fold assignments and audit proxy support.
- [ ] Confirm train and validation IDs are disjoint.
- [ ] Check image/box/mask shape, dtype, range, and label mapping.
- [ ] Replace the proxy with group-aware or multilabel splitting when the real data requires it.
- [ ] Run the test cases.